# Tiny-GPT: GPT-5 Quality Mode (Hyperparameter Sweep + Full 1.87 GB TinyStories Training)

**Instructions:**
1. Set **Runtime -> Change runtime type -> T4 GPU**.
2. Update `REPO_URL` below to your GitHub repo URL.
3. Run Stages 1-3 first (`RUN_FINAL_FULL_TRAIN = False`) to run the 4-config sweep and print the comparison table.
4. Paste the printed sweep table into chat so we pick the best config, then set `RUN_FINAL_FULL_TRAIN = True` and `FINAL_CONFIG` to train on the full 1.87 GB dataset!

In [ ]:
# === USER SETTINGS ===
REPO_URL = "https://github.com/nandish029/tiny-gpt.git"  # Change if using your fork
BRANCH = "feature/gpt4-architecture-experiments"

RUN_SWEEP = True
RUN_FINAL_FULL_TRAIN = False  # Set True after picking the sweep winner
FINAL_CONFIG = "experiments/gpt5/configs/sweep_medium.yaml"
FINAL_STEPS = 30000
DRIVE_OUT_DIR = "/content/drive/MyDrive/tiny-gpt-results"

In [ ]:
# === STAGE 1: GPU CHECK & GOOGLE DRIVE MOUNT ===
import os, torch
assert torch.cuda.is_available(), "ERROR: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU."
print(f"GPU: {torch.cuda.get_device_name(0)} | CUDA: {torch.version.cuda} | PyTorch: {torch.__version__}")

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_OUT_DIR, exist_ok=True)
print(f"Results will be saved to: {DRIVE_OUT_DIR}")

In [ ]:
# === STAGE 2: CLONE REPO & INSTALL DEPS ===
%cd /content
!rm -rf tiny-gpt
!git clone --branch {BRANCH} {REPO_URL} tiny-gpt
%cd /content/tiny-gpt
!pip -q install datasets tokenizers pyyaml numpy

In [ ]:
# === STAGE 3: PREPARE SWEEP DATASET (100k TRAIN + 2k VAL AS uint16 .bin) & RUN SWEEP ===
import json, zipfile
from src.experiments.runner import ExperimentRunner
from src.generation.generate import generate

if RUN_SWEEP:
    !python -u src/data/prepare_gpt5.py --num_train 100000 --num_val 2000 --write_bin --tokenizer_path data/gpt5/gpt5_tokenizer.json --data_dir data/gpt5_sweep

    sweep_configs = [
        "experiments/gpt5/configs/sweep_small.yaml",
        "experiments/gpt5/configs/sweep_medium.yaml",
        "experiments/gpt5/configs/sweep_medium_lr3e4.yaml",
        "experiments/gpt5/configs/sweep_ctx256.yaml",
    ]

    sweep_summary = []
    for cfg_path in sweep_configs:
        print(f"\n=== RUNNING {cfg_path} ===")
        runner = ExperimentRunner(cfg_path, device="cuda")
        params = runner.count_parameters()["trainable"]
        res = runner.train(overwrite=True)
        ev = runner.evaluate(batch_count=50, split="val", seed=10042)
        gen = torch.Generator(device="cuda").manual_seed(42)
        sample = generate(runner.model, runner.tokenizer, "Once upon a time", max_new_tokens=40, temperature=0.8, generator=gen)
        row = {
            "name": runner.name,
            "params": params,
            "elapsed_s": res["elapsed_seconds"],
            "final_train_loss": res["final_train_loss"],
            "min_val_loss": res["min_val_loss"],
            "eval_loss": ev["loss"],
            "eval_ppl": ev["perplexity"],
            "sample": sample,
        }
        sweep_summary.append(row)

    with open("experiments/gpt5/results/sweep_summary.json", "w") as f:
        json.dump(sweep_summary, f, indent=2)

    print("\n================ COPY THIS SWEEP TABLE TO CHAT ================")
    print(f"{'Name':<20} | {'Params':>9} | {'Time(s)':>7} | {'MinVal':>7} | {'EvalLoss':>8} | {'EvalPPL':>8}")
    print("-" * 74)
    for r in sweep_summary:
        print(f"{r['name']:<20} | {r['params']:>9,} | {r['elapsed_s']:>7.1f} | {r['min_val_loss']:>7.4f} | {r['eval_loss']:>8.4f} | {r['eval_ppl']:>8.4f}")
        print(f"  Sample: {r['sample']}\n")

    sweep_zip = f"{DRIVE_OUT_DIR}/gpt5_sweep_results.zip"
    with zipfile.ZipFile(sweep_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write("experiments/gpt5/results/sweep_summary.json")
    print(f"Saved sweep summary to {sweep_zip}")

In [ ]:
# === STAGE 4: FINAL FULL 1.87 GB TINYSTORIES TRAINING ===
if RUN_FINAL_FULL_TRAIN:
    import yaml, zipfile, json
    !python -u src/data/prepare_gpt5.py --num_train -1 --num_val -1 --write_bin --tokenizer_path data/gpt5/gpt5_tokenizer.json --data_dir data/gpt5_full

    with open(FINAL_CONFIG, "r") as f:
        cfg = yaml.safe_load(f)
    cfg["experiment"]["name"] = "gpt5_full"
    cfg["data"]["data_dir"] = "data/gpt5_full"
    cfg["training"]["train_steps"] = FINAL_STEPS
    cfg["training"]["val_interval"] = 500
    cfg["paths"]["checkpoint_dir"] = "checkpoints/gpt5_full"
    cfg["paths"]["results_dir"] = "experiments/gpt5/results"

    full_cfg_path = "experiments/gpt5/configs/gpt5_full_run.yaml"
    with open(full_cfg_path, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)

    runner = ExperimentRunner(full_cfg_path, device="cuda")
    res = runner.train(overwrite=True)
    ev = runner.evaluate(batch_count=100, split="val", seed=10042)
    print("FINAL TRAIN:", json.dumps(res, indent=2))
    print("FINAL EVAL:", json.dumps(ev, indent=2))

    samples = {}
    for p in ["Once upon a time", "The magic", "One day, "]:
        gen = torch.Generator(device="cuda").manual_seed(42)
        samples[p] = generate(runner.model, runner.tokenizer, p, max_new_tokens=80, temperature=0.8, generator=gen)
        print(f"[{p}] -> {samples[p]}\n")

    with open("experiments/gpt5/results/gpt5_full_eval.json", "w") as f:
        json.dump({"train": res, "eval": ev, "samples": samples}, f, indent=2)

    out_zip = f"{DRIVE_OUT_DIR}/gpt5_colab_results.zip"
    with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as zf:
        zf.write(runner.checkpoint_path)
        zf.write("experiments/gpt5/results/gpt5_full_eval.json")
        zf.write("data/gpt5_full/dataset_manifest.json")
        zf.write(full_cfg_path)
    print(f"DONE! Download {out_zip} from Google Drive and place it in your tiny-gpt folder.")